<a href="https://colab.research.google.com/github/Rakshita-Gummat/pulmonary-nodule-classification/blob/main/Assignment_done.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving assignment_cases.csv to assignment_cases.csv


In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("assignment_cases.csv", encoding="latin1")
df.head()

,Case_ID,Type,Report,Your_Risk_Category,Your_Nodule_Type,Your_Nodule_Size,Your_Nodule_Count,Your_Category_or_Guideline,Your_Management_Recommendation,Your_Reasoning
0,F-1,Fleischner,PATIENT INFORMATION: Age: 42 Gender: Female MR...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F-2,Fleischner,"PATIENT INFORMATION: - Name: Davis, Margaret -...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F-3,Fleischner,"PATIENT INFORMATION: Name: Smith, John Age: 52...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F-4,Fleischner,PATIENT INFORMATION: * Patient Name: JOHNSON...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F-5,Fleischner,PATIENT INFORMATION: - Name: George Wilson - A...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def extract_size(report):

    sizes = re.findall(r'(\d+\.?\d*)\s*mm', report)

    if sizes:
        return float(sizes[0])

    return None

In [5]:
def detect_nodule_type(report):

    report = report.lower()

    if "ground-glass" in report:
        return "GGN"

    if "part-solid" in report:
        return "Part-Solid"

    if "solid" in report:
        return "Solid"

    return "Solid"

In [6]:
def detect_count(report):

    report = report.lower()

    if "multiple" in report or "numerous" in report or "scattered" in report:
        return "Multiple"

    return "Single"

In [7]:
def detect_risk(report):

    report = report.lower()

    if ("smoker" in report or
        "tobacco" in report or
        "emphysema" in report or
        "asbestos" in report or
        "spiculated" in report):

        return "High Risk"

    return "Low Risk"

In [11]:
def fleischner_guideline(size, risk):

    if size is None:

        return (
            "No measurable nodule",
            "No routine follow-up",
            "Nodule size not detected in report"
        )

    if size < 6:

        if risk == "Low Risk":

            return (
                "Single Solid Nodule <6 mm",
                "No routine follow-up",
                "Small nodule in low-risk patient"
            )

        else:

            return (
                "Single Solid Nodule <6 mm High Risk",
                "Optional CT at 12 months",
                "Small nodule but patient has risk factors"
            )

    elif 6 <= size <= 8:

        if risk == "Low Risk":

            return (
                "Single Solid Nodule 6-8 mm Low Risk",
                "CT at 6-12 months",
                "Intermediate sized nodule"
            )

        else:

            return (
                "Single Solid Nodule 6-8 mm High Risk",
                "CT at 6-12 months and 18-24 months",
                "Intermediate nodule with risk factors"
            )

    else:

        return (
            "Solid Nodule >8 mm",
            "CT at 3 months, PET/CT, or biopsy",
            "Large suspicious nodule"
        )

In [9]:
def lung_rads(size):

    if size is None:

        return (
            "Category 1 - Negative",
            "12-month LDCT",
            "No suspicious nodules"
        )

    if size < 6:

        return (
            "Category 2 - Benign",
            "12-month LDCT",
            "Small benign appearing nodule"
        )

    elif 6 <= size < 8:

        return (
            "Category 3 - Probably Benign",
            "6-month LDCT",
            "Intermediate sized nodule"
        )

    elif 8 <= size < 15:

        return (
            "Category 4A - Suspicious",
            "3-month LDCT",
            "Suspicious nodule"
        )

    else:

        return (
            "Category 4B - Very Suspicious",
            "Diagnostic CT, PET/CT, biopsy",
            "Large suspicious lesion"
        )

In [13]:
df.head()

,Case_ID,Type,Report,Your_Risk_Category,Your_Nodule_Type,Your_Nodule_Size,Your_Nodule_Count,Your_Category_or_Guideline,Your_Management_Recommendation,Your_Reasoning
0,F-1,Fleischner,PATIENT INFORMATION: Age: 42 Gender: Female MR...,High Risk,GGN,4.0,Single,Single Solid Nodule <6 mm High Risk,Optional CT at 12 months,Small nodule but patient has risk factors
1,F-2,Fleischner,"PATIENT INFORMATION: - Name: Davis, Margaret -...",High Risk,GGN,5.0,Single,Single Solid Nodule <6 mm High Risk,Optional CT at 12 months,Small nodule but patient has risk factors
2,F-3,Fleischner,"PATIENT INFORMATION: Name: Smith, John Age: 52...",High Risk,Solid,7.0,Single,Single Solid Nodule 6-8 mm High Risk,CT at 6-12 months and 18-24 months,Intermediate nodule with risk factors
3,F-4,Fleischner,PATIENT INFORMATION: * Patient Name: JOHNSON...,High Risk,Solid,7.0,Single,Single Solid Nodule 6-8 mm High Risk,CT at 6-12 months and 18-24 months,Intermediate nodule with risk factors
4,F-5,Fleischner,PATIENT INFORMATION: - Name: George Wilson - A...,High Risk,Solid,9.0,Single,Solid Nodule >8 mm,"CT at 3 months, PET/CT, or biopsy",Large suspicious nodule


In [14]:
for i,row in df.iterrows():

    report = row["Report"]
    case_type = row["Type"]

    size = extract_size(report)
    nodule_type = detect_nodule_type(report)
    count = detect_count(report)

    if case_type == "Fleischner":

        risk = detect_risk(report)

        category, management, reasoning = fleischner_guideline(size, risk)

        df.loc[i,"Your_Risk_Category"] = risk
        df.loc[i,"Your_Nodule_Type"] = nodule_type
        df.loc[i,"Your_Nodule_Size"] = size
        df.loc[i,"Your_Nodule_Count"] = count
        df.loc[i,"Your_Category_or_Guideline"] = category
        df.loc[i,"Your_Management_Recommendation"] = management
        df.loc[i,"Your_Reasoning"] = reasoning

    else:

        category, management, reasoning = lung_rads(size)

        df.loc[i,"Your_Category_or_Guideline"] = category
        df.loc[i,"Your_Management_Recommendation"] = management
        df.loc[i,"Your_Reasoning"] = reasoning

In [15]:
df.to_csv("Pulmonary_Nodule_Assignment_Final.csv",index=False)

In [16]:
files.download("Pulmonary_Nodule_Assignment_Final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>